# Lab 3: CI/CD Pipelines & Quality Gates

## Learning Objectives
By the end of this lab, you will:
- Build four **custom DeepEval metrics** by extending `BaseMetric` (keyword coverage, latency budget, safety patterns, semantic relevance)
- Combine them with a built-in LLM-judge metric in a single `evaluate()` call
- Write a complete GitHub Actions pipeline YAML with an AI evaluation gate
- Aggregate `evaluate()` output into a release-gate decision (pass rate >= 0.90)

## Overview
Traditional CI/CD checks if code runs correctly. AI CI/CD must also check if the *system reasons correctly*. A prompt refactor might pass all unit tests while causing response relevance to drop from 0.92 to 0.74 -- only an evaluation suite catches this.

Module 03 L2 §F introduced DeepEval and its built-in LLM-judge metrics. This lab adds production-grade pieces those don't cover:
- **Deterministic, free** gates (keyword coverage, latency budget, safety patterns, cosine-similarity relevance) that fail fast before the expensive LLM judge runs
- A **release-gate aggregator** that turns `evaluate()` output into a single PASS/FAIL signal for the CI job

## What You'll Build
- Four custom DeepEval metrics — `KeywordCoverageMetric`, `LatencyBudgetMetric`, `SafetyPatternsMetric`, `SemanticRelevanceMetric`
- A 5-query golden dataset evaluated by `evaluate()` with all four custom metrics
- A GitHub Actions `deploy.yml` with quality gate blocking deployment
- Part 3 extends the metric stack with the `ContextualRelevancyMetric` LLM judge


In [ ]:
!pip install -r requirements.txt -q

---
## Part 1: Build Custom DeepEval Metrics (40 min)

DeepEval ships LLM-judge metrics (covered in Module 03 L2 §F). The CI gate adds four checks the built-ins do not cover *cheaply enough to run on every commit*:

| Metric | Why custom | Cost |
|---|---|---|
| `KeywordCoverageMetric`  | Deterministic check that the response mentions the right entities | Free |
| `LatencyBudgetMetric`    | Per-query latency SLO -- not a quality signal at all | Free |
| `SafetyPatternsMetric`   | Fast pattern pre-filter before paying for an LLM-judge safety pass | Free |
| `SemanticRelevanceMetric`| Cosine smoke test for relevance regressions -- no API key needed | Free |

Each metric subclasses `deepeval.metrics.BaseMetric` and implements one method (`measure`). Cheap deterministic metrics run first; the expensive LLM-judge metric (Part 3) runs only on test cases that survive them.

> **Convention:** every metric below reads its inputs from `tc.additional_metadata` using direct subscript access (no fallbacks). A misconfigured CI test case should error loudly, not silently pass.


In [ ]:
import numpy as np
from deepeval.metrics import BaseMetric
from deepeval.test_case import LLMTestCase


class KeywordCoverageMetric(BaseMetric):
    """Deterministic: fraction of expected_keywords found in actual_output."""

    def __init__(self, threshold: float = 0.7):
        self.threshold = threshold
        self.score = None
        self.success = None
        self.reason = None
        self.error = None
        self.evaluation_cost = 0.0

    def measure(self, tc: LLMTestCase) -> float:
        """TODO:
        1. Read tc.additional_metadata["expected_keywords"] (list of strings)
        2. Count case-insensitive substring hits in tc.actual_output
        3. self.score = hits / len(keywords) (or 1.0 if no keywords)
        4. self.success = (self.score >= self.threshold)
        5. self.reason = f"{hits}/{len(keywords)} keywords matched"
        6. Return self.score
        """
        pass  # YOUR CODE HERE

    async def a_measure(self, tc: LLMTestCase) -> float:
        return self.measure(tc)

    def is_successful(self) -> bool:
        return self.success

    @property
    def __name__(self):
        return "Keyword Coverage"


class LatencyBudgetMetric(BaseMetric):
    """Deterministic: latency_ms <= max_latency_ms.

    Reads both values from tc.additional_metadata.
    Score is 1.0 if within budget, else max_latency_ms / latency_ms (clamped 0–1).
    """

    def __init__(self):
        self.threshold = 1.0     # always-on gate (binary score)
        self.score = None
        self.success = None
        self.reason = None
        self.error = None
        self.evaluation_cost = 0.0

    def measure(self, tc: LLMTestCase) -> float:
        """TODO:
        1. Read latency_ms and max_latency_ms from tc.additional_metadata
        2. self.success = (latency_ms <= max_latency_ms)
        3. self.score = 1.0 if success else max(0.0, max_latency_ms / latency_ms)
        4. self.reason = f"{latency_ms:.0f}ms vs budget {max_latency_ms:.0f}ms"
        5. Return self.score
        """
        pass  # YOUR CODE HERE

    async def a_measure(self, tc: LLMTestCase) -> float:
        return self.measure(tc)

    def is_successful(self) -> bool:
        return self.success

    @property
    def __name__(self):
        return "Latency Budget"


class SafetyPatternsMetric(BaseMetric):
    """Deterministic: fail if any UNSAFE_PATTERNS substring appears in actual_output."""

    UNSAFE_PATTERNS = [
        "how to make a bomb", "illegal weapons", "hack into",
        "steal credentials", "bypass security",
    ]

    def __init__(self):
        self.threshold = 1.0     # binary: 1.0 = safe, 0.0 = unsafe
        self.score = None
        self.success = None
        self.reason = None
        self.error = None
        self.evaluation_cost = 0.0

    def measure(self, tc: LLMTestCase) -> float:
        """TODO:
        1. Lowercase tc.actual_output
        2. Find any matching pattern in self.UNSAFE_PATTERNS
        3. If none matched: success=True, score=1.0, reason="no unsafe patterns"
           Otherwise:        success=False, score=0.0, reason=f"matched: {pattern!r}"
        4. Return self.score
        """
        pass  # YOUR CODE HERE

    async def a_measure(self, tc: LLMTestCase) -> float:
        return self.measure(tc)

    def is_successful(self) -> bool:
        return self.success

    @property
    def __name__(self):
        return "Safety Patterns"


class SemanticRelevanceMetric(BaseMetric):
    """Deterministic: cosine similarity between tc.input and tc.actual_output embeddings.

    WARNING: this is a COARSE smoke test -- it rewards lexical/topical overlap,
    not factual correctness. Use as a regression detector ("something changed"),
    not as ground truth for relevance. Part 3 layers ContextualRelevancyMetric
    (LLM judge) on top for the sharp signal.
    """

    def __init__(self, threshold: float = 0.40):
        self.threshold = threshold
        self.score = None
        self.success = None
        self.reason = None
        self.error = None
        self.evaluation_cost = 0.0
        self._encoder = None

    @property
    def encoder(self):
        """[PROVIDED] Cached SentenceTransformer instance."""
        if self._encoder is None:
            from utils import get_encoder
            self._encoder = get_encoder()
        return self._encoder

    def measure(self, tc: LLMTestCase) -> float:
        """TODO:
        1. Encode tc.input and tc.actual_output via self.encoder.encode(..., normalize_embeddings=True)
        2. self.score = float(np.dot(q_emb, r_emb))
        3. self.success = (self.score >= self.threshold)
        4. self.reason = f"cosine sim = {self.score:.3f} (threshold {self.threshold})"
        5. Return self.score
        """
        pass  # YOUR CODE HERE

    async def a_measure(self, tc: LLMTestCase) -> float:
        return self.measure(tc)

    def is_successful(self) -> bool:
        return self.success

    @property
    def __name__(self):
        return "Semantic Relevance"


print("Custom DeepEval metrics defined: KeywordCoverage, LatencyBudget, SafetyPatterns, SemanticRelevance")


### Exercise 1.1: Evaluate the Golden Dataset

Build `LLMTestCase` objects from the golden dataset and run `evaluate()` with your four custom metrics. Then aggregate the per-case results into a release-gate decision.


In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase


GOLDEN_DATASET = [
    {
        "query": "What is retrieval-augmented generation?",
        "expected_keywords": ["retrieval", "generation", "documents"],
        "max_latency_ms": 3000,
        "simulated_response": (
            "Retrieval-Augmented Generation (RAG) combines a retrieval system with a language "
            "model. It first retrieves relevant documents from a knowledge base, then uses "
            "those documents as context for generation."
        ),
        "sim_latency_ms": 1200,
    },
    {
        "query": "How does semantic caching work?",
        "expected_keywords": ["embedding", "similarity", "cache"],
        "max_latency_ms": 3000,
        "simulated_response": (
            "Semantic caching stores query embeddings alongside responses. When a new query "
            "arrives, it computes embedding similarity against cached entries. If similarity "
            "exceeds a threshold, the cached response is returned."
        ),
        "sim_latency_ms": 800,
    },
    {
        "query": "What is a Docker multi-stage build?",
        "expected_keywords": ["docker", "stage", "build", "image"],
        "max_latency_ms": 3000,
        "simulated_response": (
            "A multi-stage Docker build uses multiple FROM instructions. The builder stage "
            "installs compilers. The runtime stage copies only compiled artifacts, resulting "
            "in a much smaller final image."
        ),
        "sim_latency_ms": 950,
    },
    {
        "query": "Explain CI/CD for AI applications",
        "expected_keywords": ["pipeline", "evaluation", "quality", "deploy"],
        "max_latency_ms": 5000,
        "simulated_response": (
            "AI CI/CD adds an evaluation stage that runs a golden dataset through the system, "
            "measuring response quality, latency, and safety. The pipeline blocks deployment "
            "if quality gates fail."
        ),
        "sim_latency_ms": 2100,
    },
    {
        "query": "What Prometheus metrics should I expose for an LLM API?",
        "expected_keywords": ["counter", "histogram", "latency", "tokens"],
        "max_latency_ms": 3000,
        "simulated_response": (
            "Key Prometheus metrics: request Counter (by model/status), latency Histogram "
            "(p50/p95/p99 buckets), token Counter (input/output), active_requests Gauge. "
            "Expose via /metrics endpoint."
        ),
        "sim_latency_ms": 1500,
    },
]


def to_test_case(entry: dict) -> LLMTestCase:
    """[PROVIDED] Pack a golden-dataset entry into an LLMTestCase.

    additional_metadata carries the deterministic-metric inputs that DeepEval's
    built-in LLMTestCase fields don't model directly.
    """
    return LLMTestCase(
        input=entry["query"],
        actual_output=entry["simulated_response"],
        retrieval_context=[
            s.strip() for s in entry["simulated_response"].split(".") if s.strip()
        ],
        additional_metadata={
            "expected_keywords": entry["expected_keywords"],
            "latency_ms":        entry["sim_latency_ms"],
            "max_latency_ms":    entry["max_latency_ms"],
        },
    )


def gate_decision(eval_results, pass_threshold: float = 0.90) -> dict:
    """[PROVIDED] Aggregate per-case metric results into a release-gate verdict."""
    test_results = eval_results.test_results
    passed = [
        r for r in test_results
        if all(m.success for m in r.metrics_data)
    ]
    pass_rate = len(passed) / len(test_results)
    return {
        "total":         len(test_results),
        "passed":        len(passed),
        "pass_rate":     pass_rate,
        "gate_decision": "PASS" if pass_rate >= pass_threshold else "FAIL",
        "results":       test_results,
    }


test_cases = [to_test_case(e) for e in GOLDEN_DATASET]
custom_metrics = [
    KeywordCoverageMetric(threshold=0.7),
    LatencyBudgetMetric(),
    SafetyPatternsMetric(),
    SemanticRelevanceMetric(threshold=0.40),
]

eval_results = evaluate(test_cases=test_cases, metrics=custom_metrics)
report = gate_decision(eval_results, pass_threshold=0.90)

print("Custom Metric Evaluation Report")
print("=" * 60)
print(f"Pass rate:     {report['pass_rate']:.1%} ({report['passed']}/{report['total']})")
print(f"Gate decision: {report['gate_decision']}")
print()
print("Per-query metric breakdown:")
for i, r in enumerate(report["results"]):
    case_pass = all(m.success for m in r.metrics_data)
    status = "PASS" if case_pass else "FAIL"
    print(f"  Q{i+1} [{status}] {r.input[:55]}...")
    for m in r.metrics_data:
        flag = "ok" if m.success else "FAIL"
        print(f"      [{flag}] {m.name}: score={m.score:.2f} -- {m.reason}")

assert report["pass_rate"] >= 0.80, f"Expected most cases to pass, got {report['pass_rate']:.1%}"
print()
print("Custom metric stack basic check passed!")


### Exercise 1.2: Add a Deliberately Failing Test Case


In [ ]:
# TODO: Build a test case that deliberately fails at least one custom metric.
# Hint: each custom metric is independently fail-able:
#   - KeywordCoverage: respond without the expected keywords
#   - LatencyBudget:   set sim_latency_ms > max_latency_ms
#   - SafetyPatterns:  include an UNSAFE_PATTERNS substring
#   - SemanticRelevance: write an off-topic response (no lexical overlap with the query)

failing_entry = {
    "query": "What is semantic caching?",
    "expected_keywords": ["embedding", "similarity", "cache"],
    "max_latency_ms": 3000,
    # TODO: Write an off-topic response (will fail KeywordCoverage AND SemanticRelevance)
    "simulated_response": "",     # e.g. "The weather in Paris is sunny today."
    # TODO: Set a latency that breaches the budget (> max_latency_ms)
    "sim_latency_ms": 0,          # e.g. 7500
}

failing_case = to_test_case(failing_entry)
failing_results = evaluate(test_cases=[failing_case], metrics=custom_metrics)
failing_report  = gate_decision(failing_results, pass_threshold=0.90)

print(f"Failing case gate: {failing_report['gate_decision']}")
for m in failing_report["results"][0].metrics_data:
    flag = "ok" if m.success else "FAIL"
    print(f"  [{flag}] {m.name}: {m.reason}")

assert failing_report["gate_decision"] == "FAIL", "This case should FAIL the quality gate!"
print()
print("Quality gate correctly rejected the bad response")


---
## Part 2: Write the GitHub Actions Workflow (30 min)

Fill in the `???` placeholders. The completed pipeline should:
1. Trigger on push to `main`
2. Run lint -> test -> evaluate -> deploy in sequence
3. Block deployment if `pass_rate < 0.90`

Use the pipeline stage table from the slides as a reference.


In [ ]:
workflow_template = '''name: AI Research Assistant - CI/CD

on:
  push:
    # TODO 1: Which branch should trigger this pipeline?
    branches: [???]
  pull_request:
    branches: [main]

env:
  REGISTRY: ghcr.io
  IMAGE_NAME: ${{ github.repository }}

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      # TODO 2: Install ruff and run it on src/, then run mypy
      - run: ???
      - run: ???

  test:
    # TODO 3: This job needs lint to succeed first
    needs: ???
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -r requirements.txt
      # TODO 4: Run unit and integration tests with pytest
      - run: pytest ???

  evaluate:
    needs: test
    runs-on: ubuntu-latest
    env:
      # TODO 5: Inject the Anthropic API key from repository secrets
      ANTHROPIC_API_KEY: ${{ secrets.??? }}
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -r requirements.txt
      - name: Run AI Evaluation Suite
        id: eval
        run: |
          python scripts/evaluate_ai.py --output eval_report.json
          # TODO 6: Extract pass_rate from JSON and capture as step output
          PASS_RATE=$(python -c "import json; r=json.load(open('eval_report.json')); print(r['???'])")
          echo "pass_rate=$PASS_RATE" >> $GITHUB_OUTPUT
      - name: Quality Gate
        run: |
          # TODO 7: Fail if pass_rate < 0.90
          python -c "
          rate = float('${{ steps.eval.outputs.pass_rate }}')
          assert rate >= ???, f'Quality gate FAILED: {rate:.0%}'
          "

  deploy:
    # TODO 8: Deploy only on main branch, after evaluate passes
    needs: evaluate
    if: ???
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Build and push Docker image
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          # TODO 9: Tag with the commit SHA
          tags: ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ ??? }}
'''

import os
os.makedirs('.github/workflows', exist_ok=True)
with open('.github/workflows/deploy.yml', 'w') as f:
    f.write(workflow_template)
print("Workflow written to .github/workflows/deploy.yml")
print()
print(workflow_template)


In [ ]:
# Structural validation: parse YAML and check job names + needs chain
import re

with open('.github/workflows/deploy.yml') as f:
    yaml_text = f.read()

unfilled = yaml_text.count("???")

checks = [
    ("No unfilled ??? placeholders",          unfilled == 0),
    ("Trigger on push to main",               "branches: [main]" in yaml_text),
    ("Job 'lint' defined",                    "  lint:" in yaml_text),
    ("Job 'test' defined",                    "  test:" in yaml_text),
    ("Job 'evaluate' defined",                "  evaluate:" in yaml_text),
    ("Job 'deploy' defined",                  "  deploy:" in yaml_text),
    ("test needs lint",                       "needs: lint" in yaml_text),
    ("evaluate needs test",                   "needs: test" in yaml_text),
    ("deploy needs evaluate",                 "needs: evaluate" in yaml_text),
    ("ANTHROPIC_API_KEY from secrets",        "secrets.ANTHROPIC_API_KEY" in yaml_text),
    ("Quality gate threshold 0.90",          "0.90" in yaml_text),
    ("deploy only on main",                   "refs/heads/main" in yaml_text or "github.ref == 'refs/heads/main'" in yaml_text),
    ("Tagged with github.sha",               "github.sha" in yaml_text),
]

print("GitHub Actions Workflow Validation")
print("=" * 55)
all_pass = True
for label, result in checks:
    status = "PASS" if result else "TODO"
    if not result:
        all_pass = False
    print(f"  [{status}] {label}")

if unfilled > 0:
    print(f"\n  Found {unfilled} unfilled '???' placeholder(s) -- replace them all!")

print()
if all_pass:
    print("All checks passed! Your deploy.yml looks correct.")
else:
    print("Fix the failing checks, then re-run this cell.")


---
## Bonus: Canary Deployment Simulation


In [ ]:
import random

def simulate_canary(version: str, error_rate_pct: float,
                    mean_latency_ms: float) -> dict:
    """Simulate canary traffic and decide: PROMOTE or ROLLBACK."""
    print(f"Canary: {version} | simulating 100 requests...")
    errors, latencies = 0, []
    for _ in range(100):
        latency = max(100, random.gauss(mean_latency_ms, mean_latency_ms * 0.2))
        latencies.append(latency)
        if random.random() < error_rate_pct / 100:
            errors += 1
    obs_error_rate = errors / 100
    obs_p99 = sorted(latencies)[99]
    decision = "PROMOTE" if (obs_error_rate < 0.02 and obs_p99 < 8000) else "ROLLBACK"
    print(f"  Error rate: {obs_error_rate:.1%} (threshold < 2%)")
    print(f"  p99 latency: {obs_p99:.0f} ms (threshold < 8000 ms)")
    print(f"  Decision: {decision}")
    return {"decision": decision, "error_rate": obs_error_rate, "p99_ms": obs_p99}

print("=" * 50)
good = simulate_canary("v2.1.0-good", error_rate_pct=0.5, mean_latency_ms=1200)
print()
print("=" * 50)
bad  = simulate_canary("v2.1.0-bad",  error_rate_pct=4.0, mean_latency_ms=9500)
print()
assert good["decision"] == "PROMOTE",  "Good version should be promoted"
assert bad["decision"]  == "ROLLBACK", "Bad version should be rolled back"
print("Canary logic verified")


---
## Part 3: Extend the Metric Stack with the LLM-Judge Layer (20 min)

Your four custom metrics catch deterministic failures cheaply. **DeepEval's contextual metrics** add an LLM-judge layer that reasons about *why* a response is good or bad — turning the cosine smoke test from `SemanticRelevanceMetric` into a sharp signal.

| Metric | What it measures | Layer |
|---|---|:---:|
| `SemanticRelevanceMetric` (custom, Part 1) | Cosine sim — coarse regression detector | Deterministic |
| `ContextualRelevancyMetric` | Does the answer actually address the query? | LLM judge |
| `ContextualPrecisionMetric` | Are the retrieved chunks actually useful for the answer? | LLM judge |
| `ContextualRecallMetric` | Does the answer cover what the retrieved chunks contain? | LLM judge |

These LLM-judge metrics were introduced in **Module 03 L2 §F** — here we drop them into the same `evaluate()` call as the custom metrics. Cells below are gated on `ANTHROPIC_API_KEY`.


In [ ]:
import os

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
DEEPEVAL_AVAILABLE = bool(ANTHROPIC_API_KEY)

if DEEPEVAL_AVAILABLE:
    from deepeval.models import LiteLLMModel
    from deepeval.metrics import (
        ContextualPrecisionMetric,
        ContextualRecallMetric,
        ContextualRelevancyMetric,
    )

    judge = LiteLLMModel(model="anthropic/claude-sonnet-4-6")

    llm_judge_metrics = [
        ContextualPrecisionMetric(threshold=0.7, model=judge, include_reason=True),
        ContextualRecallMetric(threshold=0.7,    model=judge, include_reason=True),
        ContextualRelevancyMetric(threshold=0.7, model=judge, include_reason=True),
    ]
    print("LLM-judge metrics defined:")
    for m in llm_judge_metrics:
        print(f"  {m.__class__.__name__} (threshold=0.7)")
else:
    print("ANTHROPIC_API_KEY not set -- skipping LLM-judge evaluation.")
    print("Set ANTHROPIC_API_KEY in your environment and re-run to enable.")


In [ ]:
# Stack custom + LLM-judge metrics in one evaluate() call.
# Reuses the same `test_cases` list built in Part 1 -- no test-case changes needed.

if DEEPEVAL_AVAILABLE:
    # Use the first 2 cases to keep runtime + API cost reasonable
    sample_cases = test_cases[:2]
    full_metric_stack = custom_metrics + llm_judge_metrics

    full_results = evaluate(test_cases=sample_cases, metrics=full_metric_stack)
    full_report  = gate_decision(full_results, pass_threshold=0.90)

    print("Combined Stack Evaluation Report")
    print("=" * 60)
    print(f"Pass rate:     {full_report['pass_rate']:.1%} ({full_report['passed']}/{full_report['total']})")
    print(f"Gate decision: {full_report['gate_decision']}")
    print()
    for i, r in enumerate(full_report["results"]):
        case_pass = all(m.success for m in r.metrics_data)
        status = "PASS" if case_pass else "FAIL"
        print(f"  Q{i+1} [{status}] {r.input[:55]}...")
        for m in r.metrics_data:
            flag = "ok" if m.success else "FAIL"
            reason = (m.reason or "")[:90]
            print(f"      [{flag}] {m.name}: score={m.score:.2f} -- {reason}")
else:
    print("Skipped (no ANTHROPIC_API_KEY). The cell would otherwise run:")
    print()
    print("    full_metric_stack = custom_metrics + llm_judge_metrics")
    print("    full_results = evaluate(test_cases=test_cases[:2], metrics=full_metric_stack)")
    print("    full_report  = gate_decision(full_results, pass_threshold=0.90)")
    print()
    print("Each test case would carry results for 7 metrics:")
    print("  - Keyword Coverage, Latency Budget, Safety Patterns, Semantic Relevance (deterministic)")
    print("  - ContextualPrecision, ContextualRecall, ContextualRelevancy (LLM judge)")


---
## Reflection Questions

1. **Why 0.90 pass rate?** The quality gate requires 90% of golden queries to pass. What real-world scenarios justify lowering this to 0.80? When would you raise it to 0.95?

2. **Evaluation cost:** LLM-judge metrics burn tokens on every CI run. How does ordering deterministic metrics *before* the LLM judge in your metric list reduce cost? Where in the DeepEval pipeline would you add early-exit logic?

3. **Canary rollback triggers:** Besides error rate and p99 latency, name two AI-specific signals that should trigger automatic canary rollback.

4. **Cosine vs LLM-judge relevance:** Your `SemanticRelevanceMetric` and DeepEval's `ContextualRelevancyMetric` both score relevance. Give one concrete query/response pair where they would disagree — and explain which one is right.

5. **Custom-metric reach:** Name two signals where a custom metric should be backed by an *LLM call* instead of pattern/embedding matching, and explain why.


# Your answers:
# 1.

# 2.

# 3.

# 4.

# 5.



In [ ]:
# Your answers:
# 1.

# 2.

# 3.

# 4.

# 5.
